# Predicción de la duración de estancia hospitalaria

**Proyecto**: Fundamentos de Ciencia de Datos 2025-2  
**Autor**: Thomas GIRAULT  

---

## Contexto

Hoy en día los hospitales enfrentan un gran reto: saber con antelación cuánto tiempo ocupará cada paciente una cama. Predecir la duración de la estancia no solo ayuda a mejorar la atención, sino también a organizar los recursos, reducir esperas y evitar sobrecargas en épocas de alta demanda.  

En este proyecto vamos a intentar estimar la duración de la hospitalización de un paciente a partir de información básica: sexo, índice de masa corporal (IMC), historial médico, número de ingresos previos y varios datos clínicos como indicadores de laboratorio. La idea es construir un modelo que pueda apoyar la **planificación hospitalaria** y la **gestión de camas disponibles**.  

---

## Dataset

Trabajaremos con el dataset **“Hospital Length of Stay”** publicado por Microsoft en Kaggle:  
https://www.kaggle.com/datasets/aayushchou/hospital-length-of-stay-dataset-microsoft  

En el repositorio se incluirán los datos originales (tal cual descargados) y también una versión filtrada y adaptada para nuestro análisis.  

---

## Objetivos

- Explorar y entender los datos para identificar las variables más útiles.  
- Hacer un análisis descriptivo con estadísticas y visualizaciones (distribuciones, relaciones entre variables, etc.).  
- Detectar valores atípicos y decidir si se eliminan o si tienen sentido clínico.  
- Preparar una base sólida para la siguiente fase, donde se entrenará un modelo de predicción de la duración de la estancia.  

---

## Dependencias y librerías

El análisis se hará principalmente con:  
- `numpy` y `pandas` para manipulación de datos,  
- `matplotlib` y `seaborn` para visualización,  
- opcionalmente `scipy` u otras librerías si hacen falta para estadística o detección de outliers.  

Instalación rápida:  

In [ ]:
!pip install numpy pandas matplotlib seaborn scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1ère étape : le choix des variables caractéristiques

La première étape de mon projet a été de filtrer cet énorme dataset, contenant 100k muestras et un total de 28 caractéristiques. Ce sont les caractéristiques qui nous intérèssent dans cette étape. En effet, commment pouvons nous juger si une caractéristiques est intéressante à garder, à utiliser pour notre étude ou encore s'en servir pour en construire des nouvelles.

A la fin de cette étape, je dois garder environ 6 caractéristiques dont 3 continues et 3 discretes.

## Présentation des variables dans leur globalité

Avant d'expliquer les varaibles que j'ai choisis, je vais présenter toute les variables disponibles sur le dataset. Il y en à 28. La liste est la suivante : 

<p align="center">
    <img src="../datos/data_explained.png" alt="Explicación de los datos" width="900">
</p>

On peut notammenet retrouver l'id, la date d'arrivée et de départ qui, on le sait déjà, ne nous interesssera pas. Un peu plus intéressant, de l'index 5 à l'index 15 on retrouve des comorbité imoortantes qui provienne de l'ICD (developpe un peu ce que je dis la). Ces variables sont sous forme binaire : 1 pourvrai, il possède cette comorbité et 0 pour faux. Elles sont très importantes car provenenant de l'ICD blabla développe=, améliore. Ensuite, de l'index 16 à l'index 21, on retrouve des mesures sanguines sous la forme de float. Ces variables sont très interessante et on pourra detecter des anomalies dans le sang. On retrouve aussi le bmi, le pulse, et la respiration. Et enfin, le nombre de flag de comorbité non reférencé dans l'ICD, c'est un entier qui va de 0 à 10.

Il est alors très dificiel de savoir ou donner de l'oeil car enorment de données et on ne sait pas lesquels sont les bonnes à cosiir ect. Les informations sur le sang par exemple sont très intéressante mais il y en a déjà 6. cela ne l'aisssera de la place a aucune autre. Notre professeur nous a alors conseiller de regarder ce que dis la littérature et j'ai alors passer une bonne journée à chercher, voila mes recherches : 

D'apres cette thèse : https://theses.hal.science/tel-04053390/document de ... les caractéristiques les plus importances sont l'age, les diagnosctics principaux et secondiare (comorbidités), le type d'admission ou encore l'évolution au cours du séjour. Cependant, ici parmis ces caractéristiques nous disposons que du nombre de diagnostics pricnipaux et secondaire. On pourrait d'ailleurs faire une somme des commorbidité de l'index 5 à 15 pour avoir le nombre de comorbiité princiapl (ICD9). J'ai d'ailleurs vu sur Kaggle que quelqu'un conseillait fortement d'aditionner ces valeurs pour trouver des résultats sur le model.

Ensuite, j'ai décidé de choisir le BMI. En effet, d'après la littérature, cette variable est fortement associé au temps de séjour dans un hopital, (Les patients obèses sévères ou au contraire dénutris (BMI bas) ont généralement des séjours plus longs. Plusieurs études (chirurgie cardiaque, orthopédie, soins intensifs) montrent une relation en U : BMI très bas ou très haut = plus long LOS.). Tout comme, d'apres mes recherches, la fréquence respiratoire afffecte énormement la durée de séjour et je décide de la garder. Ce sont 2 valeurs continues. https://aspenjournals.onlinelibrary.wiley.com/doi/10.1177/088453360001500405 ou encore https://pubmed.ncbi.nlm.nih.gov/15681111/ ou encore ca : https://pubmed.ncbi.nlm.nih.gov/30685103/


Cependant, chose qui m'a étonné, c'est le pouls, qui lui, a moins d'impact sur le temps de séjour. En effet, on apprend lors de nos études qu'un rythme au repos est de 60bpm mais qu'il arrive, en fonction des gens et de la physiologie, que certain bate moins (~50bpm), d'autres bate plus, ect mais cela ne veut rien dire. De us, je ne trouve pas d'étude utilisant le pouls, donc je décide de le mettre de coté.

A coté de ca, je vais ajouter le rcount à notre étude, qui compte le nombre de fois ou un patient a été readmis a l'hopital dans les 180dernier jours, ce chiffre varie de 0 à 5+. Fact : Et oui, ce n'est pas trop pratique car un 5+ peut être 5 ou encore 10. mais ce sont les legislation sur la santé aux états unis qui nous empeche d'avoir ces infos, car un patient revenant +5 fois peut etre reconaissable.




| Variable                   | Tipo       | Definición                                   | Fuente |
|-----------------------------|------------|----------------------------------------------|--------|
| BMI                        | Continua   | Índice de masa corporal del paciente         | Dataset Kaggle |
| Age                        | Continua   | Edad del paciente en años                    | Dataset Kaggle |
| Hemoglobin                 | Continua   | Nivel de hemoglobina en la sangre            | Dataset Kaggle |
| Gender                     | Discreta   | Sexo del paciente (M/F)                      | Dataset Kaggle |
| Previous Admissions        | Discreta   | Número de ingresos previos                   | Dataset Kaggle |
| Comorbidity: Hypertension  | Discreta   | Presencia o ausencia de hipertensión (0/1)   | Dataset Kaggle |


| Variable                   | Tipo       | Definición                                   | Fuente |
|-----------------------------|------------|----------------------------------------------|--------|
| BMI                        | Continua   | Índice de masa corporal del paciente         | Dataset Kaggle |
| Age                        | Continua   | Edad del paciente en años                    | Dataset Kaggle |
| Hemoglobin                 | Continua   | Nivel de hemoglobina en la sangre            | Dataset Kaggle |
| Gender                     | Discreta   | Sexo del paciente (M/F)                      | Dataset Kaggle |
| Previous Admissions        | Discreta   | Número de ingresos previos                   | Dataset Kaggle |
| Comorbidity: Hypertension  | Discreta   | Presencia o ausencia de hipertensión (0/1)   | Dataset Kaggle |
